# 03 — Results Analysis
### Multimodal Disease Detection Framework

Reproduces all key figures and tables from the paper's experimental results section.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

## 1. Performance Comparison — All Configurations (Table 4)

In [ ]:
table4 = pd.DataFrame({
    'Configuration':  ['Imaging only\n(CNN)', 'EHR only\n(MLP)', 'ECG only\n(LSTM)',
                       'Imaging +\nEHR', 'Tri-modal\n(Ours)'],
    'AUC':       [0.85, 0.83, 0.80, 0.92, 0.94],
    'Accuracy':  [82.1, 80.3, 78.6, 87.5, 89.8],
    'Precision': [0.79, 0.77, 0.75, 0.88, 0.91],
    'Recall':    [0.78, 0.76, 0.72, 0.85, 0.88],
    'F1':        [0.81, 0.78, 0.76, 0.87, 0.89],
})

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# AUC comparison
bar_colors = ['#90CAF9', '#90CAF9', '#90CAF9', '#42A5F5', '#1565C0']
bars = axes[0].bar(table4['Configuration'], table4['AUC'], color=bar_colors, edgecolor='black')
axes[0].set_ylim(0.70, 1.00)
axes[0].set_ylabel('AUC-ROC', fontsize=12)
axes[0].set_title('AUC-ROC by Configuration', fontsize=12, fontweight='bold')
for bar, val in zip(bars, table4['AUC']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.2f}', ha='center', fontweight='bold')

# Multi-metric comparison
metrics_to_plot = ['Precision', 'Recall', 'F1']
x = np.arange(len(table4))
width = 0.25
metric_colors = ['#4CAF50', '#F44336', '#FF9800']

for i, (m, c) in enumerate(zip(metrics_to_plot, metric_colors)):
    axes[1].bar(x + (i-1)*width, table4[m], width, label=m, color=c, edgecolor='black', alpha=0.9)

axes[1].set_xticks(x)
axes[1].set_xticklabels(table4['Configuration'], fontsize=9)
axes[1].set_ylim(0.60, 1.00)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Precision / Recall / F1 by Configuration', fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('Table 4: Performance Comparison — Unimodal vs Multimodal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/table4_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

display(table4)

## 2. Fusion Strategy Comparison (Table 5)

In [ ]:
table5 = pd.DataFrame({
    'Fusion Method': ['Early Fusion', 'Late Fusion', 'Intermediate\nFusion (Ours)'],
    'AUC':      [0.88, 0.89, 0.94],
    'Accuracy': [84.2, 85.1, 89.8],
    'F1':       [0.83, 0.84, 0.89],
})

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(3)
width = 0.25

ax.bar(x - width, table5['AUC'], width, label='AUC', color='#2196F3', edgecolor='black')
ax.bar(x,         [a/100 for a in table5['Accuracy']], width, label='Accuracy', color='#4CAF50', edgecolor='black')
ax.bar(x + width, table5['F1'],  width, label='F1', color='#FF9800', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(table5['Fusion Method'], fontsize=11)
ax.set_ylim(0.75, 1.00)
ax.set_ylabel('Score')
ax.set_title('Table 5: Impact of Fusion Strategy', fontweight='bold', fontsize=13)
ax.legend()
ax.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5)
ax.text(2, 0.965, '★ Best', ha='center', color='green', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('../results/table5_fusion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ablation Study (Table 6)

In [ ]:
table6 = pd.DataFrame({
    'Modality Removed':       ['None (Full Model)', '– Structured EHR', '– Time-series ECG', '– Imaging'],
    'AUC':       [0.94, 0.89, 0.90, 0.88],
    'F1':        [0.89, 0.84, 0.85, 0.83],
    'Drop (%)':  [0, 5, 4, 6],
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#1565C0', '#EF9A9A', '#EF9A9A', '#EF9A9A']
bars1 = ax1.bar(table6['Modality Removed'], table6['AUC'], color=colors, edgecolor='black')
ax1.set_ylim(0.80, 0.97)
ax1.set_title('AUC — Ablation Study (Table 6)', fontweight='bold')
ax1.set_ylabel('AUC')
ax1.tick_params(axis='x', rotation=20)
for bar, val in zip(bars1, table6['AUC']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.2f}', ha='center', fontweight='bold')

ax2.bar(table6['Modality Removed'][1:], table6['Drop (%)'][1:],
        color=['#F44336', '#FF9800', '#9C27B0'], edgecolor='black')
ax2.set_title('Performance Drop (%) per Removed Modality', fontweight='bold')
ax2.set_ylabel('Relative AUC Drop (%)')
ax2.tick_params(axis='x', rotation=20)

plt.suptitle('Ablation Study — Contribution of Each Modality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/table6_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Benchmark Comparison (Table 8)

In [ ]:
table8 = pd.DataFrame({
    'Model':      ['MedFuse\n(Late Fusion)', 'MMT Transformer\n(Early Fusion)', 'Proposed Model\n(Intermediate Fusion)'],
    'AUC':        [0.90, 0.91, 0.94],
    'Precision':  [0.86, 0.88, 0.91],
    'Recall':     [0.84, 0.86, 0.88],
    'Robustness': [2, 1, 3],  # 1=Low, 2=Moderate, 3=High
})

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3)
width = 0.25

colors = ['#2196F3', '#4CAF50', '#F44336']
ax.bar(x - width, table8['AUC'], width, label='AUC', color='#2196F3', edgecolor='black')
ax.bar(x,         table8['Precision'], width, label='Precision', color='#4CAF50', edgecolor='black')
ax.bar(x + width, table8['Recall'], width, label='Recall', color='#F44336', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(table8['Model'], fontsize=10)
ax.set_ylim(0.78, 1.0)
ax.set_ylabel('Score')
ax.set_title('Table 8: Benchmark Comparison with State-of-the-Art Models', fontweight='bold', fontsize=13)
ax.legend()
ax.text(2, 0.96, '★ Proposed', ha='center', color='green', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/table8_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
display(table8.drop(columns='Robustness'))

## 5. Summary — Key Results

In [ ]:
summary = {
    'Metric': ['AUC (Tri-modal)', 'AUC (Imaging only)', 'AUC (Missing imaging)',
               'Recall (Pneumonia)', 'F1 (Tri-modal)', 'vs MedFuse AUC improvement'],
    'Value':  ['0.94', '0.85', '0.88', '88%', '0.89', '+0.04'],
    'Reference': ['Table 4', 'Table 4', 'Table 7', 'Section 5', 'Table 4', 'Table 8']
}

df_summary = pd.DataFrame(summary)
print('\n=== KEY RESULTS FROM PAPER ===')
display(df_summary.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))